In [1]:
%pip install -r ../../requirements-offline.txt --quiet

/Users/utarn/projects/ml-metrology/.claude/worktrees/issue-27/.venv/bin/python: No module named pip


Note: you may need to restart the kernel to use updated packages.


# Exercise — Module M2: Thai OCR bake-off

**วิธีทำ:** เติมช่อง `# TODO` / `...` ให้ทุกเซลล์รันผ่าน แล้วตอบคำถาม markdown ที่กำหนด
— เปิด `book.ipynb` เทียบได้ตลอด (โค้ดชุดเดียวกัน)

**Dataset:** openthaigpt/thai-ocr-evaluation — <https://huggingface.co/datasets/openthaigpt/thai-ocr-evaluation>, **License: CC BY-SA 4.0** (อยู่ใน repo: `datasets/thai_ocr_evaluation/`)

**เวลาโดยประมาณ:** 15 นาที (โมเดลโหลดจาก cache ในเครื่อง — ถ้ายังไม่มี การโหลดครั้งแรกใช้อินเทอร์เน็ต)


## 0. เตรียมข้อมูล + ผลจาก book (ให้มาแล้ว — รันผ่าน)

book.ipynb รัน bake-off 20 ภาพ (4 ภาพ/หมวด, seed 42) แล้ว — exercise ชุดนี้ใช้
**ชุดเดียวกัน** เพื่อให้เทียบกับผลใน book ได้ตรง


In [2]:
import csv  # noqa: E402
from pathlib import Path  # noqa: E402


def repo_root() -> Path:
    """หา repo root โดยไล่ขึ้นจาก cwd."""
    for p in (Path.cwd(), *Path.cwd().parents):
        if (p / "requirements-offline.txt").exists():
            return p
    raise SystemExit(
        "ไม่พบ repo root — เปิด JupyterLab จาก repo root (uv run jupyter lab)"
    )


ROOT = repo_root()
DATA = ROOT / "datasets" / "thai_ocr_evaluation" / "test"
REPORTS = ROOT / "datasets" / "measurement_reports"
TESSDATA = ROOT / "datasets" / "tessdata"
assert (DATA / "metadata.csv").exists(), (
    "ไม่พบ thai_ocr_evaluation — ดู datasets/README.md"
)
assert TESSDATA.exists(), "ไม่พบ datasets/tessdata — เช็คว่า repo ครบ"
print("repo root:", ROOT)


repo root: /Users/utarn/projects/ml-metrology/.claude/worktrees/issue-27


In [3]:
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
import jiwer  # noqa: E402, F401

meta = list(csv.DictReader(open(DATA / "metadata.csv", encoding="utf-8")))
truth = {r["file_name"]: r["text"] for r in meta}

cats: dict[str, list[str]] = {}
for r in meta:
    cats.setdefault(r["category"], []).append(r["file_name"])

rng = np.random.default_rng(42)
subset: list[tuple[str, str]] = []
for cat in sorted(cats):
    files = sorted(cats[cat])
    idx = sorted(rng.choice(len(files), 4, replace=False))
    subset += [(cat, files[i]) for i in idx]
refs = [truth[fn].strip() for _, fn in subset]
print(len(subset), "ภาพ")


20 ภาพ


In [4]:
# ผลจาก book เก็บไว้เทียบ (CER รวมของ 3 โมเดล — รันจาก book.ipynb)
BOOK_CER = {
    "Tesseract tha": 0.723,
    "EasyOCR": 0.315,
    "thai-trocr": 0.135,
}
print(BOOK_CER)


{'Tesseract tha': 0.723, 'EasyOCR': 0.315, 'thai-trocr': 0.135}


## ข้อ 1 — Tesseract: เพิ่ม `eng` แล้วดีขึ้นไหม?

ภาพเอกสารไทยจริงมักผสมอังกฤษ (ชื่อเครื่องมือ, Serial No., หน่วย) — ลอง
`tesseract lang='tha+eng'` แล้วเทียบ CER (book ใช้ `tha` เดี่ยว ได้ CER รวม ~0.72)

**คำถาม:** `tha+eng` ดีขึ้นกี่เท่า และดีขึ้นเฉพาะภาพหมวดไหน (ตอบใน markdown)


In [5]:
import pytesseract  # noqa: E402


def ocr_tesseract(path, lang="tha+eng") -> str:
    """TODO 1: OCR ไฟล์ด้วย tesseract — ชี้ --tessdata-dir ที่ TESSDATA."""
    cfg = ...  # สร้าง string f"--tessdata-dir {TESSDATA}"
    txt = pytesseract.image_to_string(str(path), lang=lang, config=cfg)
    return " ".join(txt.split())


tess2_out = {}
for cat, fn in subset:
    tess2_out[fn] = ocr_tesseract(DATA / fn, lang="tha+eng")

# TODO 1: คำนวณ CER รวมของ tess2_out เทียบ refs แล้วเก็บใน tess2_cer
tess2_cer = ...
print(f"tha+eng CER = {tess2_cer:.3f}  (book: tha เดี่ยว = 0.723)")


AttributeError: 'ellipsis' object has no attribute 'read'

In [6]:
# TODO 1 (ต่อ): CER เฉพาะภาพผสมอังกฤษ (หมวด document_enth)
enth = [(c, f) for c, f in subset if c == "document_enth"]
refs_e = [truth[f].strip() for _, f in enth]
hyps_e = [tess2_out[f] for _, f in enth]
print("document_enth CER:", ...)


KeyError: 'images/pdf_enth_1.png'

**คำตอบข้อ 1:**

_(เติมตรงนี้)_


## ข้อ 2 — EasyOCR: หมวดไหนยากสุด?

รัน EasyOCR ทั้ง 20 ภาพ (โค้ดเหมือน book) แล้วสร้างตาราง CER **แยกตามหมวด**

**คำถาม:** หมวดไหนยากสุดสำหรับ EasyOCR และเพราะอะไร (ตอบใน markdown)


In [7]:
import easyocr  # noqa: E402
from PIL import Image  # noqa: E402

reader = easyocr.Reader(["th", "en"], gpu=False, verbose=False)

easy_out = {}
for cat, fn in subset:
    img = Image.open(DATA / fn).convert("RGB")
    lines = reader.readtext(np.array(img), detail=0)
    easy_out[fn] = " ".join(lines)

# TODO 2: สร้าง DataFrame คอลัมน์ category / cer (CER รายภาพของ easy_out)
# แล้ว groupby หาค่าเฉลี่ยต่อหมวด เก็บใน easy_by_cat
rows = []
for cat, fn in subset:
    rows.append({"category": cat, "cer": ...})
easy_by_cat = pd.DataFrame(rows).groupby("category")["cer"].mean()
print(easy_by_cat.round(3))


/Users/utarn/projects/ml-metrology/.claude/worktrees/issue-27/.venv/lib/python3.14/site-packages/torch/ao/nn/quantized/dynamic/modules/rnn.py:162: UserWarning: torch.quantize_per_tensor, torch.quantize_per_channel and other quantized tensor creation functions that produce tensors with dtype torch.quint8, torch.qint8, and torch.qint32 are deprecated and will be removed in a future PyTorch release. Please see https://github.com/pytorch/pytorch/issues/184982 for more information. (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/quantized/Quantizer.cpp:116.)
  w_ih = torch.quantize_per_tensor(
/Users/utarn/projects/ml-metrology/.claude/worktrees/issue-27/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


TypeError: agg function failed [how->mean,dtype->object]

**คำตอบข้อ 2:**

_(เติมตรงนี้)_


## ข้อ 3 — Normalize ก่อนวัด CER: ตัว `กํา` ของ tesseract

tesseract มักคืน `กําแพง` (nikhahit `ํ` + `า` แยกกัน) ทั้งที่ข้อความจริงคือ `กำ`
(สระอำรวมตัว) — ตัวอักษร "เหมือนกัน" ในสายตาคน แต่ bytes ไม่เหมือน → CER
โดยรู้เทือน

TODO 3: เขียน `norm()` แทน `ํา` ด้วย `ำ` (และ `.strip()` ช่องว่าง) แล้ว**วัดใหม่**
CER ของ tesseract (`tha` เดี่ยว ใช้ผลใน book — รันซ้ำที่นี่ก็ได้) ก่อน/หลัง normalize


In [8]:
# TODO 3: รัน tesseract lang='tha' ใหม่ (เหมือน book) เก็บใน tess1_out
tess1_out = {}
for cat, fn in subset:
    tess1_out[fn] = ...

cer_before = ...
print("CER ก่อน normalize:", round(cer_before, 3))


def norm(text: str) -> str:
    """TODO 3: normalize ข้อความ OCR — แทน 'ํา' ด้วย 'ำ' + strip."""
    return ...


cer_after = ...
print("CER หลัง normalize:", round(cer_after, 3))


TypeError: type ellipsis doesn't define __round__ method

**คำตอบข้อ 3:** CER ลดลงเท่าไร — และบอกอะไรเกี่ยวกับ "การวัดผลโมเดลข้อความ"?

_(เติมตรงนี้)_


## ข้อ 4 — Surya: นับโครงสร้างหน้ารายงานการวัด

ใช้ Surya layout กับ**หน้าสรุปผล/ลายเซ็น**ของรายงานการวัด (หน้าสุดท้ายใน
`datasets/measurement_reports/`) — นับว่ามีกล่องโครงสร้างอะไรบ้าง

**คำถาม:** หน้านี้มี layout label อะไรบ้าง แต่ละ label มีกี่กล่อง (ตอบใน markdown)


In [9]:
from surya.layout import LayoutPredictor  # noqa: E402

layout_pred = LayoutPredictor()

# TODO 4: เลือกไฟล์หน้าสุดท้าย (sorted(REPORTS.glob('*.png'))[-1])
# รัน layout แล้วนับจำนวนกล่องต่อ label ด้วย pd.Series(...).value_counts()
page = Image.open(...).convert("RGB")
layouts = layout_pred([page])[0]
print(...)


/Users/utarn/projects/ml-metrology/.claude/worktrees/issue-27/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


AttributeError: 'ellipsis' object has no attribute 'read'

**คำตอบข้อ 4:**

_(เติมตรงนี้)_


## สรุป (เติมหลังทำครบ)

- โมเดลที่ CER ดีที่สุดใน lab นี้: ______
- ปัจจัยที่ทำให้ CER "จริง" ต่างจากที่วัด: ______
- สิ่งที่จะใช้ต่อใน Capstone Track C (อ่านค่าจากสแกนรายงานการวัด): ______
